# Visium Spatial Transcriptomics Analysis with liana+

This notebook demonstrates a complete analysis workflow for 10x Genomics Visium data using squidy and liana+.

#### Setup and Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('/home/jungj2/project/lung/ipf_visium_ng')

import sys
# 현재 환경에 맞는 경로 사용
sys.path.append('/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp')  # mcp_servers 상위 폴더 

In [3]:
import numpy as np
import pandas as pd
# pip install "numpy<2.0" anndata==0.10.5.post1 scanpy==1.9.8 
import scanpy as sc
import squidpy as sq

import omnipath as op

import decoupler as dc
import liana as li

import seaborn as sns
from matplotlib import pyplot as plt

# set dpi to 50, to make the notebook smaller
# plt.rcParams['figure.dpi'] = 100
from utils.utils_lianap import *

/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will b

In [4]:
import os                                                                         
os.chdir('/home/jungj2/project/lung/ipf_visium_ng')   

## 0. data prep

In [5]:
raw_df_meta = pd.read_csv("res/meta.csv")
raw_df_meta

,sample,indi,disease,age,gender,description,class,rep,name
0,V19S23-092-A1,HC_1,normal,44.0,female,"B0, replicate 1",B0,rep1,B0 (HC_1/rep1)
1,V10T03-282-A1,HC_1,normal,44.0,female,"B0, replicate 2",B0,rep2,B0 (HC_1/rep2)
2,V10T31-015-A1,HC_2,normal,66.0,male,"B0, replicate 1",B0,rep1,B0 (HC_2/rep1)
3,V10T31-019-A1,HC_3,normal,56.0,male,"B0, replicate 1",B0,rep1,B0 (HC_3/rep1)
4,V10T03-280-A1,HC_3,normal,56.0,male,"B0, replicate 2",B0,rep2,B0 (HC_3/rep2)
5,V10T03-281-A1,HC_4,normal,62.0,male,"B0, replicate 2",B0,rep2,B0 (HC_4/rep2)
6,V19S23-092-B1,IPF_1,idiopathic pulmonary fibrosis,50.0,male,"B1, replicate 1",B1,rep1,B1 (IPF_1/rep1)
7,V10T03-279-B1,IPF_1,idiopathic pulmonary fibrosis,50.0,male,"B1, replicate 2",B1,rep2,B1 (IPF_1/rep2)
8,V10T31-015-B1,IPF_2,idiopathic pulmonary fibrosis,57.0,female,"B1, replicate 1",B1,rep1,B1 (IPF_2/rep1)
9,V10T03-280-B1,IPF_2,idiopathic pulmonary fibrosis,57.0,female,"B1, replicate 2",B1,rep2,B1 (IPF_2/rep2)


In [6]:
# df_meta = raw_df_meta.query("rep == 'rep1'")
# df_meta = raw_df_meta.query("sample in ['V19S23-092-A1', 'V19S23-092-B1']")
df_meta = raw_df_meta

In [7]:
SAMPLE_NAMES = df_meta['sample']
# sample_name = 'V19S23-092-B1'

In [8]:
import squidpy as sq
adata_list = []

for sample_name in SAMPLE_NAMES:
  print(sample_name)
  data_path = f'/common/jungj2/project/data/visium/ipf_lung/data/{sample_name}'
  # adata = sc.read_visium(data_path)
  adata = sq.read.visium(data_path)
  
  adata.obs_names = [f"{sample_name}_{bc}" for bc in adata.obs_names]

  adata.var_names_make_unique()
  adata.layers['counts'] = adata.X.copy()
  sc.pp.normalize_total(adata, target_sum=1e4)
  sc.pp.log1p(adata)
  adata_list.append(adata)


V19S23-092-A1


/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


V10T03-282-A1


/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


V10T31-015-A1


...

/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


V10T03-281-D1


/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


V10T31-051-D1


/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/home/jungj2/project/lung/ipf_visium_ng/sp_enrich_mcp/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


In [9]:
adata_list[0].obs

,in_tissue,array_row,array_col
V19S23-092-A1_AAACAACGAATAGTTC-1,1,0,16
V19S23-092-A1_AAACAAGTATCTCCCA-1,1,50,102
V19S23-092-A1_AAACAATCTACTAGCA-1,1,3,43
V19S23-092-A1_AAACACCAATAACTGC-1,1,59,19
V19S23-092-A1_AAACAGAGCGACTCCT-1,1,14,94
...,...,...,...
V19S23-092-A1_TTGTTCAGTGTGCTAC-1,1,24,64
V19S23-092-A1_TTGTTCTAGATACGCT-1,1,21,3
V19S23-092-A1_TTGTTGTGTGTCAAGA-1,1,31,77
V19S23-092-A1_TTGTTTCACATCCAGG-1,1,58,42


In [10]:
# 한 번에 concat
adata = sc.concat(
    adata_list, 
    label='sample',           # obs에 샘플 레이블 추가
    # keys=SAMPLE_NAMES,        # 인덱스 prefix로 사용
    join='inner',             # 공통 유전자만 (권장)
    # index_unique='-'          # 중복 방지
)
adata.obs['sample'] = adata.obs.index.str.split('_').str[0]
adata.obs

,in_tissue,array_row,array_col,sample
V19S23-092-A1_AAACAACGAATAGTTC-1,1,0,16,V19S23-092-A1
V19S23-092-A1_AAACAAGTATCTCCCA-1,1,50,102,V19S23-092-A1
V19S23-092-A1_AAACAATCTACTAGCA-1,1,3,43,V19S23-092-A1
V19S23-092-A1_AAACACCAATAACTGC-1,1,59,19,V19S23-092-A1
V19S23-092-A1_AAACAGAGCGACTCCT-1,1,14,94,V19S23-092-A1
...,...,...,...,...
V10T31-051-D1_TTGTTTCACATCCAGG-1,1,58,42,V10T31-051-D1
V10T31-051-D1_TTGTTTCATTAGTCTA-1,1,60,30,V10T31-051-D1
V10T31-051-D1_TTGTTTCCATACAACT-1,1,45,27,V10T31-051-D1
V10T31-051-D1_TTGTTTGTATTACACG-1,1,73,41,V10T31-051-D1


In [11]:
label_mapping = dict(zip(df_meta['sample'], df_meta['name']))
label_mapping

# Add sample name
adata.obs['type'] = adata.obs['sample'].map(label_mapping).astype('category')
adata.obs

,in_tissue,array_row,array_col,sample,type
V19S23-092-A1_AAACAACGAATAGTTC-1,1,0,16,V19S23-092-A1,B0 (HC_1/rep1)
V19S23-092-A1_AAACAAGTATCTCCCA-1,1,50,102,V19S23-092-A1,B0 (HC_1/rep1)
V19S23-092-A1_AAACAATCTACTAGCA-1,1,3,43,V19S23-092-A1,B0 (HC_1/rep1)
V19S23-092-A1_AAACACCAATAACTGC-1,1,59,19,V19S23-092-A1,B0 (HC_1/rep1)
V19S23-092-A1_AAACAGAGCGACTCCT-1,1,14,94,V19S23-092-A1,B0 (HC_1/rep1)
...,...,...,...,...,...
V10T31-051-D1_TTGTTTCACATCCAGG-1,1,58,42,V10T31-051-D1,B3 (IPF_4/rep1)
V10T31-051-D1_TTGTTTCATTAGTCTA-1,1,60,30,V10T31-051-D1,B3 (IPF_4/rep1)
V10T31-051-D1_TTGTTTCCATACAACT-1,1,45,27,V10T31-051-D1,B3 (IPF_4/rep1)
V10T31-051-D1_TTGTTTGTATTACACG-1,1,73,41,V10T31-051-D1,B3 (IPF_4/rep1)


In [12]:
tot_df_meta = pd.read_csv("res/IRIS/meta_IRIS_15.csv")
tot_df_meta
matched_spots = adata.obs.index[adata.obs.index.isin(tot_df_meta['spotName'])]
print(f"Matched spots: {len(matched_spots)} / {adata.n_obs}")
adata = adata[matched_spots, :].copy()
df_meta = tot_df_meta.set_index('spotName').loc[matched_spots]
print((df_meta.index.values == adata.obs.index.values).all())
adata.obs['IRIS_domain'] = (
      df_meta['IRIS_domain']
      .astype(int)
      .astype(str)
      .astype('category')
  )
adata.obs

Matched spots: 101806 / 101907
True


,in_tissue,array_row,array_col,sample,type,IRIS_domain
V19S23-092-A1_AAACAACGAATAGTTC-1,1,0,16,V19S23-092-A1,B0 (HC_1/rep1),11
V19S23-092-A1_AAACAAGTATCTCCCA-1,1,50,102,V19S23-092-A1,B0 (HC_1/rep1),11
V19S23-092-A1_AAACAATCTACTAGCA-1,1,3,43,V19S23-092-A1,B0 (HC_1/rep1),9
V19S23-092-A1_AAACACCAATAACTGC-1,1,59,19,V19S23-092-A1,B0 (HC_1/rep1),11
V19S23-092-A1_AAACAGAGCGACTCCT-1,1,14,94,V19S23-092-A1,B0 (HC_1/rep1),11
...,...,...,...,...,...,...
V10T31-051-D1_TTGTTTCACATCCAGG-1,1,58,42,V10T31-051-D1,B3 (IPF_4/rep1),8
V10T31-051-D1_TTGTTTCATTAGTCTA-1,1,60,30,V10T31-051-D1,B3 (IPF_4/rep1),8
V10T31-051-D1_TTGTTTCCATACAACT-1,1,45,27,V10T31-051-D1,B3 (IPF_4/rep1),8
V10T31-051-D1_TTGTTTGTATTACACG-1,1,73,41,V10T31-051-D1,B3 (IPF_4/rep1),8


In [13]:
tot_IRIS = pd.read_csv("res/IRIS/prop_IRIS_15.csv")
compositions = tot_IRIS.set_index('spotName').loc[adata.obs.index]
print((compositions.index.values == adata.obs.index.values).all())
compositions = compositions.drop(columns=['sample', 'barcode', 'x', 'y'])
compositions

adata.obsm['compositions'] = compositions
adata.obsm['compositions'] = adata.obsm['compositions'].fillna(0)                   
adata.obsm['compositions']

True


,MUC5AC+ High,Basal,Proliferating Epithelial Cells,Differentiating Ciliated,Ciliated,SCGB3A2+ SCGB1A1+,SCGB3A2+,MUC5B+,KRT5-/KRT17+,Transitional AT2,...,B Cells,Monocytes,T Cells,Macrophages,PLIN2+ Fibroblasts,Fibroblasts,HAS1 High Fibroblasts,Myofibroblasts,Mesothelial Cells,Smooth Muscle Cells
V19S23-092-A1_AAACAACGAATAGTTC-1,8.476954e-07,0.003995,0.012227,0.004072,0.009077,0.006995,0.037787,4.723146e-04,0.007428,0.089846,...,0.030326,0.084933,0.000358,0.104801,6.980930e-03,0.018183,1.279578e-04,0.115283,1.409879e-04,0.041258
V19S23-092-A1_AAACAAGTATCTCCCA-1,1.967046e-03,0.003420,0.013284,0.008695,0.012947,0.010714,0.032197,3.253295e-03,0.009428,0.086976,...,0.036064,0.067199,0.002468,0.111416,2.167079e-04,0.007510,3.787926e-06,0.120310,9.315358e-06,0.016652
V19S23-092-A1_AAACAATCTACTAGCA-1,1.189873e-02,0.011324,0.011055,0.020422,0.032030,0.023975,0.043311,1.416733e-02,0.005378,0.073110,...,0.037744,0.089257,0.000660,0.133406,5.633171e-04,0.007874,6.015313e-06,0.075175,9.848233e-05,0.023661
V19S23-092-A1_AAACACCAATAACTGC-1,1.991970e-04,0.002458,0.016413,0.003238,0.010252,0.008477,0.025631,2.144671e-03,0.004329,0.087472,...,0.027768,0.075338,0.001612,0.086895,4.412729e-03,0.016783,6.846483e-08,0.132674,1.478125e-04,0.038284
V19S23-092-A1_AAACAGAGCGACTCCT-1,2.909342e-04,0.003871,0.017724,0.038539,0.070569,0.015611,0.036862,3.142012e-03,0.007039,0.067171,...,0.030688,0.074085,0.002894,0.110046,3.906857e-03,0.012167,8.040203e-06,0.091707,1.223615e-05,0.025752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
V10T31-051-D1_TTGTTTCACATCCAGG-1,9.685527e-08,0.000025,0.006220,0.000043,0.000330,0.000219,0.006579,2.803526e-06,0.016644,0.040066,...,0.025750,0.001230,0.000128,0.096393,1.111289e-05,0.000072,1.788219e-01,0.351403,9.414447e-08,0.076920
V10T31-051-D1_TTGTTTCATTAGTCTA-1,6.026703e-04,0.000102,0.005796,0.000129,0.000317,0.000270,0.007883,3.126839e-04,0.015054,0.043500,...,0.029136,0.004874,0.000138,0.107927,2.873227e-07,0.000006,1.842131e-01,0.311194,2.015763e-09,0.065943
V10T31-051-D1_TTGTTTCCATACAACT-1,3.228571e-07,0.000005,0.003909,0.000005,0.000102,0.000002,0.002785,1.820714e-06,0.010499,0.034300,...,0.026626,0.000698,0.000001,0.081625,4.384270e-08,0.000002,2.321990e-01,0.376861,2.690148e-09,0.066013
V10T31-051-D1_TTGTTTGTATTACACG-1,3.241014e-04,0.000086,0.005495,0.000103,0.000139,0.000206,0.004334,1.504773e-04,0.018078,0.042655,...,0.025593,0.002070,0.000021,0.106041,3.992306e-07,0.000007,1.931415e-01,0.342674,1.269543e-07,0.065687


In [14]:
set(adata.obs['IRIS_domain'])

{'0',
 '1',
 '10',
 '11',
 '12',
 '13',
 '14',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9'}

In [15]:
adata.obs.columns = adata.obs.columns.str.replace('/', '_', regex=False)
# obsm 내 DataFrame 컬럼명 수정
for key in adata.obsm.keys():  
    if hasattr(adata.obsm[key], 'columns'):  # DataFrame인 경우
        adata.obsm[key].columns = adata.obsm[key].columns.str.replace('/', '_', regex=False)
        
adata.write_h5ad('sp_enrich_mcp/data/n_ipf.h5ad')
# adata = sc.read_h5ad('sp_enrich_mcp/data/n_ipf.h5ad')
# adata

## 0. set param

In [16]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import decoupler as dc
import liana as li
from matplotlib import pyplot as plt

# # set dpi to 50, to make the notebook smaller
# plt.rcParams['figure.dpi'] = 100

# adata = sc.read_h5ad('data/n_ipf.h5ad')
cluster_key = 'IRIS_domain'
data_name = 'n_ipf'
save_path = 'sp_enrich_mcp/res_ipf'

sample_key = 'type'


## 1. Spatial cluster plot

In [17]:
from mcp_servers.runplot_basic import plot_cluster
plot_cluster(
    filename=f'sp_enrich_mcp/data/{data_name}.h5ad',
    cluster_key=cluster_key,    
    sample_key = sample_key,
    save_path = save_path
    )

Spatial plot saved to: sp_enrich_mcp/res_ipf/1.cluster_spatial_by_type.png


## 2. cell type
### 2-1. Summary of celll type with spatial cluster

In [18]:
from mcp_servers.runplot_basic import runplot_clusterwithcelltyp
runplot_clusterwithcelltyp(
    filename=f'sp_enrich_mcp/data/{data_name}.h5ad',
    cluster_key=cluster_key,
    show_significance=True,
    save_path = save_path
    )

Starting cell type composition analysis...
1. Extracting data from anndata...
   Using spot-based mode with obsm['compositions']
   Detected 31 cell types from compositions
2. Extracting cluster labels from adata.obs['IRIS_domain']...
3. Calculating cluster median proportions...
4. Performing statistical tests (using FDR-adjusted p-values)...
5. Creating heatmap with significance markers...
Heatmap saved to: sp_enrich_mcp/res_ipf/2.celltype_heatmap_analysis.png

Analysis complete!


## 3. neighborhood analysis
### 3-1. neighborhood enrichment analysis (using squdiy)

Computing a neighborhood enrichment can help us identify spots clusters that share a common neighborhood structure across the tissue. We can compute such score with the following function: squidpy.gr.nhood_enrichment(). In short, it’s an enrichment score on spatial proximity of clusters: if spots belonging to two different clusters are often close to each other, then they will have a high score and can be defined as being enriched. On the other hand, if they are far apart, and therefore are seldom a neighborhood, the score will be low and they can be defined as depleted. This score is based on a permutation-based test, and you can set the number of permutations with the n_perms argument (default is 1000). 

#### NOTE!!! we need to set the **cluster_key**, **source_cluster**, **target_cluster** parameter.

In [19]:
from mcp_servers.runplot_NeighborEnrich import plot_neighborhood_enrichment
plot_neighborhood_enrichment(
    filename=f'sp_enrich_mcp/data/{data_name}.h5ad',
    cluster_key=cluster_key,
    save_path = save_path
    )

   Removed 6100 long edges (>99.0th percentile)
   Platform: single_cell | Spatial neighbors: Delaunay + 2-ring expansion


100%|██████████| 1000/1000 [00:07<00:00, 139.68/s]


Neighborhood enrichment plot saved to: sp_enrich_mcp/res_ipf/3.neighborhood_enrichment_squidpy.png


In [20]:
# sq.gr.nhood_enrichment(adata, cluster_key=cluster_key)
adata

AnnData object with n_obs × n_vars = 101806 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'type', 'IRIS_domain'
    obsm: 'spatial', 'compositions'
    layers: 'counts'

## 4. pathway enrichment analysis using Hallmakr gene set(decoupler)

Let’s estimate pathway activities as a way to make the data a bit more interpretable. We will use decoupler-py with pathways genesets from PROGENy. See this tutorial for details.


In [21]:
from mcp_servers.runplot_PathwayEnrich import runplot_pathway_enrich
runplot_pathway_enrich(
    filename=f'sp_enrich_mcp/data/{data_name}.h5ad',
    cluster_key=cluster_key,
    organism = 'human',
    geneset= 'progeny',
    reference = 'rest',
    threshold = 0.05,
    n_markers = 3,
    save_path = save_path
    )

Spatial Enrichment analysis (decoupler) plot saved to: sp_enrich_mcp/res_ipf/4.spatial_enrich_decoupler.png


## 5. MISTy (Spatial Relationships)

In [22]:
from mcp_servers.runplot_SpatialRelation import runplot_spatial_relationship

runplot_spatial_relationship(
    filename=f'sp_enrich_mcp/data/{data_name}.h5ad',
    cluster_key=cluster_key,
    save_path=save_path
    )

   Using spot-based mode with obsm['compositions']
   Detected 31 cell types from compositions
Spatial relationship (MISTy) plot saved to: sp_enrich_mcp/res_ipf/5.spatial_relation_MISTy.png
